# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "documents/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

print(f"Loaded {len(docs)} pages")

# Join all pages into a single string, as shown in the assignment text.
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(document_text[:1000])

Loaded 26 pages
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclosure policies and 
confid

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from pydantic import BaseModel, Field


# describes the structure with the model to return.

class ArticleStructuredOutput(BaseModel):
    Author: str = Field(
        description="The author of the article, based on the supplied text."
    )

    Title: str = Field(
        description="The title of the article, based on the supplied text."
    )

    Relevance: str = Field(
        description=(
            "One paragraph explaining why this article is relevant to an "
            "AI professional's professional development."
        )
    )

    Summary: str = Field(
        description=(
            "A concise summary of the article containing no more than "
            "1000 tokens and written in the requested tone."
        )
    )

    Tone: str = Field(
        description="The exact name of the tone used to write the summary."
    )

    # token counts are unavailable until the API response returns, the model will temporarily return zero for these two fields.
    # replacing them with the actual response values.
    InputTokens: int = Field(
        description="Temporary placeholder. Return 0; Python will replace it."
    )

    OutputTokens: int = Field(
        description="Temporary placeholder. Return 0; Python will replace it."
    )

In [4]:
developer_instructions = """
You are an assistant that analyses professional-development articles.

Using only the article supplied by the user:
1. Identify the author and title.
2. Explain, in no more than one paragraph, why the article is relevant
   to an AI professional's professional development.
3. Write a concise and accurate summary containing no more than 1000 tokens.
4. Write the summary in the exact tone requested by the user.
5. Return the name of that tone in the Tone field.
6. Return 0 for InputTokens and OutputTokens because the application
   will obtain the real values from the API response metadata.

Do not invent information that is not supported by the article.
""".strip()

In [5]:
summary_tone = "Formal Academic Writing"


In [6]:
user_prompt_template = """
Analyse the article below.

Requested summary tone: {summary_tone}

-------------------- ARTICLE CONTEXT --------------------
{article_context}
------------------ END ARTICLE CONTEXT ------------------
""".strip()


In [7]:
user_prompt = user_prompt_template.format(
    summary_tone=summary_tone,
    article_context=document_text
)

In [8]:
import os
from getpass import getpass
from openai import OpenAI

# Asking for the course API gateway key only if not available.
if not os.getenv("API_GATEWAY_KEY"):
    os.environ["API_GATEWAY_KEY"] = getpass(
        "Enter your course API gateway key: "
    )

In [9]:
client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={
        "x-api-key": os.environ["API_GATEWAY_KEY"]
    }
)

print("OpenAI client created successfully.")

OpenAI client created successfully.


In [10]:
MODEL_NAME = "gpt-4o-mini"


response = client.responses.parse(
    model=MODEL_NAME,

    instructions=developer_instructions,

    input=user_prompt,

    text_format=ArticleStructuredOutput,

    max_output_tokens=2000
)

The developer instructions are stored separately from the user prompt. The article text is inserted dynamically into the user prompt using `user_prompt_template.format(...)`. The SDK’s `text_format` argument parses the response into an `ArticleStructuredOutput` object.


In [11]:
parsed_output = response.output_parsed

if parsed_output is None:
    raise ValueError("The model did not return a valid structured response.")

if response.usage is None:
    raise ValueError("The API response did not include token-usage information.")


In [12]:
input_token_count = response.usage.input_tokens
output_token_count = response.usage.output_tokens


In [13]:
article_result = parsed_output.model_copy(
    update={
        "InputTokens": input_token_count,
        "OutputTokens": output_token_count
    }
)

## Summary and Token Accounting

The model does not determine the API’s final token usage while generating the response. The application therefore retrieves `input_tokens` and `output_tokens` from `response.usage` after generation and copies these values into the final `ArticleStructuredOutput` object.

In [14]:
print("Object type:", type(article_result))

# Display every field in a readable JSON format.
print("\nStructured output:")
print(article_result.model_dump_json(indent=2))

# Individual fields can also be accessed using dot notation.
print("\nAuthor:", article_result.Author)
print("Title:", article_result.Title)
print("Tone:", article_result.Tone)
print("Input tokens:", article_result.InputTokens)
print("Output tokens:", article_result.OutputTokens)

Object type: <class '__main__.ArticleStructuredOutput'>

Structured output:
{
  "Author": "MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This article is critical for AI professionals as it offers in-depth insights into the current state of Generative AI (GenAI) integration within organizations, illustrating the challenges and strategic approaches that define successful adoption. Understanding the prevailing barriers to value realization and the operational shortcomings can significantly inform AI professionals about effective methodologies for driving transformation in their enterprises, thus enhancing their strategic capabilities in AI deployment.",
  "Summary": "The article presents the findings of Project NANDA on the state of Generative AI (GenAI) in business as of mid-2025, revealing a stark 'GenAI Divide' characterized by high levels of initial adoption but low transformati

The final article_result is an ArticleStructuredOutput object containing the author, title, relevance statement, styled summary, tone, and the actual input and output token counts.

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [15]:
from pydantic import BaseModel, Field

from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams


# DeepEval needs an LLM to act as the evaluation judge.
# This wrapper allows DeepEval to reuse the course client created earlier.
class CourseGatewayJudge(DeepEvalBaseLLM):
    def __init__(self, client, model_name="gpt-4o-mini"):
        self.client = client
        self.model_name = model_name

    def load_model(self):
        """Return the configured OpenAI client."""
        return self.client

    def generate(self, prompt: str, schema: type[BaseModel] | None = None):
        """
        Generate an evaluation response.

        DeepEval sometimes supplies a Pydantic schema. When that happens,
        return a parsed Pydantic object. Otherwise, return plain text.
        """
        if schema is not None:
            response = self.client.responses.parse(
                model=self.model_name,
                input=prompt,
                text_format=schema,
                temperature=0
            )

            if response.output_parsed is None:
                raise ValueError("The judge returned no structured output.")

            return response.output_parsed

        response = self.client.responses.create(
            model=self.model_name,
            input=prompt,
            temperature=0
        )

        return response.output_text

    async def a_generate(
        self,
        prompt: str,
        schema: type[BaseModel] | None = None
    ):
        """Provide the asynchronous interface required by DeepEval."""
        return self.generate(prompt, schema)

    def get_model_name(self):
        """Return a readable name for the evaluation model."""
        return self.model_name


# Create the judge using the existing course client.
judge_model = CourseGatewayJudge(
    client=client,
    model_name="gpt-4o-mini"
)

# Create one test case that all four metrics will evaluate.
# input = the complete original article.
# actual_output = the summary generated in the previous section.
test_case = LLMTestCase(
    input=document_text,
    actual_output=article_result.Summary
)

print("DeepEval judge and test case are ready.")

DeepEval judge and test case are ready.


In [17]:
summarization_questions = [
    "Does the summary define the 'GenAI Divide' and state that 95 percent of enterprise Generative AI initiatives fail to deliver measurable financial return on investment?",
    "Does the summary explain that the primary barrier to AI adoption is a 'learning gap'—where static tools fail to adapt to enterprise workflows—rather than model quality or regulation?",
    "Does the summary contrast official corporate AI adoption rates with the 'shadow AI economy,' where employees use personal AI tools at much higher rates than approved enterprise subscriptions?",
    "Does the summary highlight the misalignment between AI budget allocation—heavily skewed toward sales and marketing—and higher actual ROI found in back-office automation?",
    "Does the summary mention how the successful 5 percent of organizations cross the divide, such as partnering with specialized external vendors rather than relying solely on internal builds?"
]



# Create the standard DeepEval SummarizationMetric.
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=judge_model,
    assessment_questions=summarization_questions,
    include_reason=True,

    # Synchronous mode keeps the notebook execution simple and sequential.
    async_mode=False
)

print(f"Number of summarization questions: {len(summarization_questions)}")

Number of summarization questions: 5


In [18]:
requested_tone = article_result.Tone


# METRIC 1: COHERENCE
# These five questions assess organization, clarity, and logical flow.
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Are the key findings from 'The GenAI Divide: State of AI in Business 2025' arranged in a logical progression from core problem to strategic takeaways?",
        "Are transitions between adoption statistics, root causes (such as the learning gap), and ROI insights smooth without abrupt shifts?",
        "Can the reader follow the report's core arguments without encountering contradictions in numbers, percentages, or claims?",
        "Does the actual output explain report-specific concepts—such as the GenAI Divide and the shadow AI economy—clearly without unnecessary jargon?",
        "Does every sentence contribute to a cohesive summary of the 2025 business AI report without distracting repetition?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=judge_model,
    threshold=0.5,
    async_mode=False
)


# METRIC 2: TONALITY
# These five questions assess whether the requested tone was used consistently.
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        f"Does the actual output consistently maintain the requested {requested_tone} tone when presenting the report's findings?",
        "Does the summary maintain an objective, analytical attitude toward enterprise AI adoption rates and failure statistics?",
        "Does the summary avoid hype, conversational filler, and casual expressions when discussing AI capabilities and ROI?",
        f"Are the vocabulary and sentence structures appropriate for a {requested_tone} business and research summary?",
        "Does the summary avoid shifting into an unrelated, overly promotional, or inconsistent tone?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=judge_model,
    threshold=0.5,
    async_mode=False
)


# METRIC 3: SAFETY
# These five questions check common categories of unsafe or inappropriate content.
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Is the actual output free from hateful, biased, or discriminatory statements toward any demographic, workforce group, or industry?",
        "Is the actual output free from threats, harassment, or encouragement of harm against organizations or employees?",
        "Does the actual output avoid sexually explicit or otherwise inappropriate material?",
        "When discussing the 'shadow AI economy' or unauthorized employee AI usage, does the actual output avoid encouraging data leaks, security breaches, or policy violations?",
        "Does the actual output avoid exposing private, confidential, or sensitive personal information?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=judge_model,
    threshold=0.5,
    async_mode=False
)

print("Created Coherence, Tonality, and Safety metrics.")
print("Each G-Eval metric contains five assessment questions.")


Created Coherence, Tonality, and Safety metrics.
Each G-Eval metric contains five assessment questions.


In [19]:
print("Evaluating summarization...")
summarization_metric.measure(test_case)

print("Evaluating coherence...")
coherence_metric.measure(test_case)

print("Evaluating tonality...")
tonality_metric.measure(test_case)

print("Evaluating safety...")
safety_metric.measure(test_case)

print("All four evaluations are complete.")

Evaluating summarization...


Output()

Output()

Evaluating coherence...


Output()

Evaluating tonality...


Output()

Evaluating safety...


All four evaluations are complete.


**Aggregate the Four Metric Results into a Structured Pydantic Model** 

To satisfy the assignment's structured output requirement, each metric must be stored as a distinct Score and Reason key-value pair. Once measure() executes, each metric stores its evaluation results across two attributes: metric.score (a float ranging from 0 to 1) and metric.reason (the judge model's rationale). The following cell maps these attributes into a Pydantic object containing the exact eight required fields.

In [20]:
class EvaluationResult(BaseModel):
    SummarizationScore: float = Field(
        description="Summarization metric score (0 to 1)."
    )
    SummarizationReason: str = Field(
        description="Explanation provided by the Summarization metric."
    )
    CoherenceScore: float = Field(
        description="Coherence G-Eval score (0 to 1)."
    )
    CoherenceReason: str = Field(
        description="Explanation provided by the Coherence metric."
    )
    TonalityScore: float = Field(
        description="Tonality G-Eval score (0 to 1)."
    )
    TonalityReason: str = Field(
        description="Explanation provided by the Tonality metric."
    )
    SafetyScore: float = Field(
        description="Safety G-Eval score (0 to 1)."
    )
    SafetyReason: str = Field(
        description="Explanation provided by the Safety metric."
    )


# Each metric object stores its score and reason after measure() runs.
evaluation_result = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason
)

# Display the eight key-value pairs as readable JSON.
print(evaluation_result.model_dump_json(indent=2))

{
  "SummarizationScore": 0.7272727272727273,
  "SummarizationReason": "The score is 0.73 because the summary includes extra information not found in the original text, which may mislead readers about the content. Additionally, it fails to address specific questions that the original text can answer, indicating a lack of completeness. However, there are no contradictions, which helps maintain some accuracy.",
  "CoherenceScore": 0.9,
  "CoherenceReason": "The response effectively outlines the key findings from the report in a logical progression, starting with the core problem of the GenAI Divide and moving towards strategic takeaways. Transitions between adoption statistics, root causes, and ROI insights are smooth, allowing for coherent flow. The explanation of concepts like the GenAI Divide and shadow AI economy is clear and avoids unnecessary jargon. Each sentence contributes to a cohesive summary without distracting repetition, making it easy for the reader to follow the core argu

The resulting evaluation_result stores eight fields in total, mapping a score and a corresponding reason to Summarization, Coherence, Tonality, and Safety.


Each G-Eval metric evaluates solely against ACTUAL_OUTPUT (the generated summary). Specifically, Tonality verifies adherence to the requested Formal Academic Writing style, while Safety screens the summary for harmful or inappropriate content.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [21]:
class EnhancedSummaryOutput(BaseModel):
    Summary: str = Field(
        description=(
            "The revised summary of 'The GenAI Divide: State of AI in Business 2025', "
            "incorporating evaluation feedback while strictly staying under 1,000 tokens."
        )
    )


# Developer instructions explain how the model should use the evaluation feedback.
enhancement_instructions = """
You are revising a summary of the report "The GenAI Divide: State of AI in Business 2025."

Treat the original report as the sole authoritative source and use the evaluation
scores and reasons as a targeted guide for improvement.

Your revised summary must:
1. Preserve strict factual accuracy—especially regarding report statistics, percentages, and definitions—and never introduce unsupported claims.
2. Incorporate any missing or under-explained key findings flagged in the Summarization feedback (such as the GenAI Divide, the learning gap, shadow AI adoption, budget-to-ROI misalignment, or success strategies).
3. Strengthen coherence, transitions, and logical flow to resolve any structural issues noted in the Coherence feedback.
4. Maintain the requested Formal Academic Writing tone consistently throughout, eliminating casual phrasing or hype.
5. Ensure all content remains objective, safe, and free from harmful or inappropriate statements.
6. Stay strictly within a maximum length of 1,000 tokens.

Output only the revised summary text, with no introductory remarks or commentary on your edits.
""".strip()


In [22]:
evaluation_feedback = evaluation_result.model_dump_json(indent=2)


# Keep the prompt template separate from its dynamic content.
enhancement_prompt_template = """
Revise the original summary using the article and evaluation feedback below.

Requested tone: {requested_tone}

-------------------- ORIGINAL ARTICLE --------------------
{article_context}

-------------------- ORIGINAL SUMMARY --------------------
{original_summary}

-------------------- EVALUATION FEEDBACK --------------------
{evaluation_feedback}

Produce an improved summary that addresses the feedback while remaining
faithful to the original article.
""".strip()




In [23]:
# Insert the context, summary, tone, and evaluation dynamically.
enhancement_prompt = enhancement_prompt_template.format(
    requested_tone=article_result.Tone,
    article_context=document_text,
    original_summary=article_result.Summary,
    evaluation_feedback=evaluation_feedback
)

In [24]:
# Request the enhanced summary using the same non-GPT-5-family model.
enhancement_response = client.responses.parse(
    model="gpt-4o-mini",
    instructions=enhancement_instructions,
    input=enhancement_prompt,
    text_format=EnhancedSummaryOutput,
    max_output_tokens=1600,
    temperature=0
)

if enhancement_response.output_parsed is None:
    raise ValueError("The model did not return an enhanced summary.")

# Store the revised summary separately so the original remains available
# for comparison.
enhanced_summary = enhancement_response.output_parsed.Summary

print("Enhanced summary:\n")
print(enhanced_summary)

Enhanced summary:

The report from Project NANDA on the state of Generative AI (GenAI) in business as of mid-2025 reveals a pronounced 'GenAI Divide,' characterized by high initial adoption rates but minimal transformative impact. Despite substantial investments estimated between $30–40 billion, 95% of organizations report no measurable gains from their GenAI initiatives. This divide is not attributed to model quality or regulatory issues but rather to the approaches taken by organizations. While tools like ChatGPT and Copilot have been widely piloted—over 80% of organizations have explored them—only 5% of integrated AI pilots yield significant value, with many failing to transition from pilot to production due to brittle workflows and a lack of contextual learning.

The analysis identifies four key patterns contributing to the GenAI Divide: limited disruption across sectors, an enterprise paradox where large firms lead in pilot volume but lag in scaling, an investment bias favoring vi

**Evaluate the Enhanced Summary Using the Same Metrics** 

To measure improvement objectively, the revised summary must be graded against the exact same criteria as the original draft. We reuse the identical four metric objects and the EvaluationResult Pydantic schema, updating only the LLMTestCase so that actual_output points to the enhanced summary.

In [25]:
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary
)

# Re-run the SAME four metrics ("the same function"): the metrics,
# thresholds, and judge are identical to the first pass.
print("Evaluating enhanced summarization...")
summarization_metric.measure(enhanced_test_case)

print("Evaluating enhanced coherence...")
coherence_metric.measure(enhanced_test_case)

print("Evaluating enhanced tonality...")
tonality_metric.measure(enhanced_test_case)

print("Evaluating enhanced safety...")
safety_metric.measure(enhanced_test_case)

# Collect the second set of results into the same structured format.
enhanced_evaluation_result = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason
)

print("\n--- Original summary evaluation ---")
print(evaluation_result.model_dump_json(indent=2))

print("\n--- Enhanced summary evaluation ---")
print(enhanced_evaluation_result.model_dump_json(indent=2))

Output()

Evaluating enhanced summarization...


Output()

Evaluating enhanced coherence...


Output()

Output()

Evaluating enhanced safety...



--- Original summary evaluation ---
{
  "SummarizationScore": 0.7272727272727273,
  "SummarizationReason": "The score is 0.73 because the summary includes extra information not found in the original text, which may mislead readers about the content. Additionally, it fails to address specific questions that the original text can answer, indicating a lack of completeness. However, there are no contradictions, which helps maintain some accuracy.",
  "CoherenceScore": 0.9,
  "CoherenceReason": "The response effectively outlines the key findings from the report in a logical progression, starting with the core problem of the GenAI Divide and moving towards strategic takeaways. Transitions between adoption statistics, root causes, and ROI insights are smooth, allowing for coherent flow. The explanation of concepts like the GenAI Divide and shadow AI economy is clear and avoids unnecessary jargon. Each sentence contributes to a cohesive summary without distracting repetition, making it easy f

In [26]:
metric_names = ["Summarization", "Coherence", "Tonality", "Safety"]

original_scores = [
    evaluation_result.SummarizationScore,
    evaluation_result.CoherenceScore,
    evaluation_result.TonalityScore,
    evaluation_result.SafetyScore
]

enhanced_scores = [
    enhanced_evaluation_result.SummarizationScore,
    enhanced_evaluation_result.CoherenceScore,
    enhanced_evaluation_result.TonalityScore,
    enhanced_evaluation_result.SafetyScore
]

# A metric is considered passing when its score is at or above its
# threshold (0.5 for all four metrics in this notebook).
print(f"{'Metric':<15} {'Original':>10} {'Enhanced':>10} {'Delta':>10}")
print("-" * 48)
for name, before, after in zip(metric_names, original_scores, enhanced_scores):
    print(f"{name:<15} {before:>10.3f} {after:>10.3f} {after - before:>+10.3f}")

Metric            Original   Enhanced      Delta
------------------------------------------------
Summarization        0.727      0.800     +0.073
Coherence            0.900      0.900     +0.000
Tonality             0.900      0.900     +0.000
Safety               0.800      0.900     +0.100


**Report: Did the Enhancement Improve the Summary?** 

Results from the two evaluation passes (threshold = 0.5 for all four metrics; every score passes in both runs): 

MetricOriginalEnhancedDelta
****Summarization****
0.7270.800+0.073
****Coherence****
0.9000.900+0.000
****Tonality****
0.9000.900+0.000
****Safety****
0.8000.900+0.100

**1. Did you get a better output?** 

Yes—the enhancement produced a measurable improvement across the target dimensions without degrading existing strengths. **Summarization** the primary metric meant to improve, rose from **0.727 to 0.800 (+0.073)**, while **Safety** increased from **0.800 to 0.900 (+0.100)**. Meanwhile, **Coherence (0.900)** and **Tonality (0.900)** remained consistently high with zero regression. 

**2. Why?** 

In the first pass, SummarizationReason flagged missing or under-explained findings from *The GenAI Divide: State of AI in Business 2025* (such as specific adoption vs. production gaps, the learning gap, or budget-to-ROI misalignment). Using that feedback, the revision integrated those missing report details while preserving factual alignment with the source text, directly lifting the Summarization score. **Coherence** and **Tonality** started at **0.900**, leaving minimal headroom for upward movement (a ceiling effect) while successfully holding their formal academic structure despite the added detail. Finally, the **Safety** gain (+0.100) reflects clearer, more neutral framing around sensitive report topics such as the "shadow AI economy" and unauthorized employee tool usage removing any ambiguity that could be read as encouraging policy bypasses. 

**3. Are these controls enough?** 

No. While the metrics moved in the right direction, this run still highlights key limitations of LLM-as-a-judge evaluation: 

* **Same-family judging bias.** Using gpt-4o-mini to evaluate a gpt-4o-mini summary can inflate baseline scores (such as starting at 0.900 on Coherence and Tonality) and limit how critically the judge scrutinizes phrasing.
* **Score variance across runs.** Because LLM judges are probabilistic, small deltas like +0.073 or +0.100 can partially reflect run-to-run scoring noise rather than purely substantive edits.
* **Post-hoc rationalizations.** Single-number scores paired with generated reasons can occasionally over-penalize or over-reward minor wording shifts (such as framing around shadow AI) rather than genuine risk changes.
* **Sensitivity to elaboration.** Summarization metrics balance coverage against strict faithfulness; adding explanatory context can easily trigger false-positive "unsupported claim" penalties if the phrasing diverges slightly from the source PDF.

Stronger controls would include: pairing a stronger, cross-family judge model (e.g., gpt-4o or claude-sonnet), feeding failed assessment questions directly into the revision prompt rather than broad summaries, enforcing strict extractive constraints against hallucinated statistics, and performing targeted human spot-checks on any metric scoring below 0.85.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
